In [6]:
import pandas as pd
import numpy as np

excel_file = '225 g of bacon on gas burner, fan off.xls'

print("1. Loading tabs from Excel...")
df_sens = pd.read_excel(excel_file, sheet_name='Sensors')
df_inst = pd.read_excel(excel_file, sheet_name='Instruments')

# Standardize Time
df_sens.rename(columns={'A - Time (s)': 'Time'}, inplace=True)
df_inst.rename(columns={'Time (s)': 'Time'}, inplace=True)

# Convert to float and round to align timestamps
df_sens['Time'] = df_sens['Time'].astype(float).round(1)
df_inst['Time'] = df_inst['Time'].astype(float).round(1)

print("2. Merging data...")
merged_df = df_sens.merge(df_inst, on='Time', how='outer')

# Fill gaps so we have a continuous timeline
merged_df = merged_df.sort_values('Time').ffill().bfill()

# 3. Labeling Logic
merged_df['class_id'] = np.where(merged_df['Time'] < 0, 0, 1)
merged_df['label'] = merged_df['class_id'].map({0: 'Normal', 1: 'Cooking', 2: 'Other'})

# 4. Final Table Mapping (Using Node A and Node B)
final_df = pd.DataFrame({
    'label': merged_df['label'],
    'class_id': merged_df['class_id'],
    'temperature': merged_df['T (deg C)'],
    'humidity': merged_df['Rel Humidity (%)'],
    'tvoc_ppb': merged_df['A - CO (ppm)'],  # Chemical signal from Node A
    'eco2_ppm': merged_df['B - CO (ppm)']   # Chemical signal from Node B (different location)
})

# 5. Numerical cleaning
cols_to_fix = ['temperature', 'humidity', 'tvoc_ppb', 'eco2_ppm']
final_df[cols_to_fix] = final_df[cols_to_fix].apply(pd.to_numeric, errors='coerce')

# Drop any broken rows
final_df = final_df.dropna(subset=['tvoc_ppb', 'eco2_ppm', 'temperature'])

# Save the result
output_file = 'NIST_cleaned_bacon.csv'
final_df.to_csv(output_file, index=False)

print(f"Success! '{output_file}' now has unique data for TVOC and eCO2 using Node A and B.")
print(final_df.head())

1. Loading tabs from Excel...
2. Merging data...
Success! 'NIST_cleaned_bacon.csv' now has unique data for TVOC and eCO2 using Node A and B.
    label  class_id  temperature  humidity  tvoc_ppb  eco2_ppm
0  Normal         0         23.8      20.8       0.0       0.0
1  Normal         0         23.8      20.8       0.0       0.0
2  Normal         0         23.8      20.8       0.0       0.0
3  Normal         0         23.8      20.8       0.0       0.0
4  Normal         0         23.8      20.8       0.0       0.0
